# Credit Risk & Policy Intelligence
Synthetic portfolio case study. Run all cells to regenerate source tables, analysis, and dashboard.
Evaluate a credit model and a hypothetical approval policy on fully matured, out-of-time outcomes.

In [1]:
from pathlib import Path
import runpy, sys, json
root = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'scripts'))
runpy.run_path(str(root / 'scripts' / 'analyze.py'), run_name='__main__')

{
  "train_rows": 5021,
  "validation_rows": 2502,
  "test_rows": 2477,
  "test_roc_auc": 0.7370122284939429,
  "test_average_precision": 0.5199122306256447,
  "test_brier_score": 0.1725993245150115,
  "test_prevalence": 0.2753330641905531,
  "policy_threshold": 0.15999999999999998,
  "test_approval_rate": 0.421881308033912,
  "test_policy_contribution": 419269.6847999999,
  "approve_all_contribution": -4930493.0602,
  "observation_cutoff": "2025-12-31"
}
{'__name__': '__main__', '__doc__': None, '__package__': '', '__loader__': None, '__spec__': None, '__file__': '/workspace/scratch/9104f481554f/projects/credit-risk-policy-intelligence/scripts/analyze.py', '__cached__': None, '__builtins__': {'__name__': 'builtins', '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explic

## Inspect validated outcomes
Training-only imputation and scaling; logistic model; chronological train/validation/test split; validation-only threshold selection; PR AUC, ROC AUC, Brier score, calibration, population stability, and group audits.

In [2]:
import pandas as pd
pd.read_csv(root / 'outputs' / 'monthly_kpis.csv').head(12)

      month        segment  ...  contribution  default_rate
0   2024-07       Contract  ...   -10810.9683      0.327586
1   2024-07       Salaried  ...    64973.2805      0.228346
2   2024-07  Self employed  ...   -30152.5755      0.303797
3   2024-08       Contract  ...    -7225.9915      0.253968
4   2024-08       Salaried  ...    16907.8031      0.269663
5   2024-08  Self employed  ...    46545.5241      0.190000
6   2024-09       Contract  ...    28255.1868      0.234043
7   2024-09       Salaried  ...   108262.9669      0.241803
8   2024-09  Self employed  ...    20031.2319      0.287356
9   2024-10       Contract  ...    14436.9223      0.428571
10  2024-10       Salaried  ...   -35516.4565      0.298932
11  2024-10  Self employed  ...      193.7455      0.285714

[12 rows x 9 columns]


In [3]:
print((root / 'outputs' / 'executive_brief.md').read_text())

# Credit risk decision brief

Synthetic educational analysis. No deployment or real lending recommendation.

- Out-of-time ROC AUC: 0.737; average precision: 0.520, compared with default prevalence 27.5%. Brier score: 0.173.
- Validation selects a PD cutoff of 0.16; the held-out test approval rate is 42.2%.
- Test policy contribution: $419,270; approve-all comparison: $-4,930,493. These depend on assumed 12% nondefault contribution and 65% loss given default, not comprehensive loan cash flows.
- Review calibration and group rejection error rates before considering any operational pilot. Group attributes are excluded from model features; this alone does not establish fairness.

All simulated applicants have outcomes, including those rejected by the hypothetical policy. Real underwriting data typically lacks rejected-applicant outcomes, so this comparison cannot directly transfer to production. The dataset contains fully matured 12-month outcomes; cutoff is December 2025. Missing-income 

## SQL cross-check
Queries are evaluated on the same generated SQLite source tables.

In [4]:
runpy.run_path(str(root / 'scripts' / 'run_sql.py'), run_name='__main__')
pd.read_csv(root / 'outputs' / 'sql_result_1.csv').head()

Query 1: 72 rows
Query 2: 3 rows
Query 3: 24 rows
  origination_month     employment  applications  default_rate     mean_loan
0           2023-01       Contract            47      0.404255  14984.009149
1           2023-01       Salaried           280      0.196429  18104.881786
2           2023-01  Self employed            95      0.326316  19723.626421
3           2023-02       Contract            57      0.280702  16648.801579
4           2023-02       Salaried           258      0.263566  18222.449419
